# 1. What are SQL Joins?

**Joins** combine rows from two or more tables based on a related column (usually a foreign key relationship).

## Why Use Joins?
- Data is normalized across multiple tables
- Need to retrieve related information together
- Avoid data duplication
- Answer complex queries spanning multiple tables

## Types of Joins

| Join Type | Description |
|-----------|-------------|
| `INNER JOIN` | Returns only matching rows from both tables |
| `LEFT JOIN` | Returns all from left table + matching from right |
| `RIGHT JOIN` | Returns all from right table + matching from left |
| `FULL OUTER JOIN` | Returns all rows when match in either table |
| `CROSS JOIN` | Returns Cartesian product of both tables |
| `SELF JOIN` | Table joins with itself |

# 2. Setup - Sample Database

In [ ]:
import mysql.connector
from mysql.connector import Error

DB_CONFIG = {
    'host': 'localhost',
    'user': 'root',
    'password': 'your_password',
    'database': 'joins_demo'
}

def get_connection(with_db=True):
    config = DB_CONFIG.copy()
    if not with_db:
        del config['database']
    return mysql.connector.connect(**config)

def execute_query(query, fetch=False):
    """Execute a query and optionally fetch results."""
    conn = get_connection()
    cursor = conn.cursor(dictionary=True)
    cursor.execute(query)
    if fetch:
        result = cursor.fetchall()
        cursor.close()
        conn.close()
        return result
    conn.commit()
    cursor.close()
    conn.close()

def print_results(query, title=""):
    """Execute query and print formatted results."""
    if title:
        print(f"\n{title}")
        print("=" * 70)
    results = execute_query(query, fetch=True)
    if results:
        # Print header
        headers = results[0].keys()
        print(" | ".join(f"{h:15}" for h in headers))
        print("-" * 70)
        # Print rows
        for row in results:
            print(" | ".join(f"{str(v):15}" for v in row.values()))
    else:
        print("No results found.")
    return results

print("Helper functions created!")

In [ ]:
# Create database and tables for join demonstrations

conn = get_connection(with_db=False)
cursor = conn.cursor()

# Create database
cursor.execute("DROP DATABASE IF EXISTS joins_demo")
cursor.execute("CREATE DATABASE joins_demo")
cursor.execute("USE joins_demo")

# Create tables
cursor.execute("""
    CREATE TABLE departments (
        dept_id INT PRIMARY KEY AUTO_INCREMENT,
        dept_name VARCHAR(50) NOT NULL
    )
""")

cursor.execute("""
    CREATE TABLE employees (
        emp_id INT PRIMARY KEY AUTO_INCREMENT,
        name VARCHAR(100) NOT NULL,
        dept_id INT,
        manager_id INT,
        salary DECIMAL(10,2),
        FOREIGN KEY (dept_id) REFERENCES departments(dept_id),
        FOREIGN KEY (manager_id) REFERENCES employees(emp_id)
    )
""")

cursor.execute("""
    CREATE TABLE projects (
        project_id INT PRIMARY KEY AUTO_INCREMENT,
        project_name VARCHAR(100) NOT NULL,
        budget DECIMAL(12,2)
    )
""")

cursor.execute("""
    CREATE TABLE employee_projects (
        emp_id INT,
        project_id INT,
        role VARCHAR(50),
        PRIMARY KEY (emp_id, project_id),
        FOREIGN KEY (emp_id) REFERENCES employees(emp_id),
        FOREIGN KEY (project_id) REFERENCES projects(project_id)
    )
""")

conn.commit()
print("Tables created!")
cursor.close()
conn.close()

In [ ]:
# Insert sample data

conn = get_connection()
cursor = conn.cursor()

# Departments
cursor.executemany("""
    INSERT INTO departments (dept_name) VALUES (%s)
""", [
    ("Engineering",),
    ("Marketing",),
    ("HR",),
    ("Finance",),
    ("Research",)  # Department with no employees
])

# Employees (some without department)
cursor.executemany("""
    INSERT INTO employees (name, dept_id, manager_id, salary) VALUES (%s, %s, %s, %s)
""", [
    ("Alice Johnson", 1, None, 95000),    # Manager in Engineering
    ("Bob Smith", 1, 1, 75000),           # Reports to Alice
    ("Carol White", 1, 1, 72000),         # Reports to Alice
    ("David Brown", 2, None, 85000),      # Manager in Marketing
    ("Eva Martinez", 2, 4, 65000),        # Reports to David
    ("Frank Wilson", 3, None, 70000),     # HR
    ("Grace Lee", 4, None, 90000),        # Finance
    ("Henry Taylor", None, None, 50000),  # No department (contractor)
])

# Projects
cursor.executemany("""
    INSERT INTO projects (project_name, budget) VALUES (%s, %s)
""", [
    ("Website Redesign", 50000),
    ("Mobile App", 120000),
    ("Data Migration", 80000),
    ("Marketing Campaign", 30000),
])

# Employee-Project assignments
cursor.executemany("""
    INSERT INTO employee_projects (emp_id, project_id, role) VALUES (%s, %s, %s)
""", [
    (1, 1, "Lead"),
    (1, 2, "Advisor"),
    (2, 1, "Developer"),
    (2, 2, "Developer"),
    (3, 2, "Developer"),
    (4, 4, "Lead"),
    (5, 4, "Coordinator"),
    (7, 3, "Lead"),
])

conn.commit()
print("Sample data inserted!")
cursor.close()
conn.close()

In [ ]:
# View our sample data

print_results("SELECT * FROM departments", "DEPARTMENTS")
print_results("SELECT * FROM employees", "EMPLOYEES")
print_results("SELECT * FROM projects", "PROJECTS")
print_results("SELECT * FROM employee_projects", "EMPLOYEE-PROJECT ASSIGNMENTS")

# 3. INNER JOIN

Returns only rows that have matching values in **both** tables.

```
Table A         Table B
+---+          +---+
| 1 |          | 1 |  <- Match
| 2 |          | 3 |  
| 3 |          | 4 |  <- Match (3)
+---+          +---+

INNER JOIN Result: 1, 3 (only matches)
```

## Syntax
```sql
SELECT columns
FROM table1
INNER JOIN table2 ON table1.column = table2.column
```

In [ ]:
# INNER JOIN: Employees with their departments
# Only returns employees who HAVE a department

query = """
SELECT 
    e.emp_id,
    e.name,
    d.dept_name,
    e.salary
FROM employees e
INNER JOIN departments d ON e.dept_id = d.dept_id
"""

print_results(query, "INNER JOIN: Employees WITH Departments")
print("\nNote: Henry Taylor (no dept) is NOT included!")

In [ ]:
# INNER JOIN with multiple tables
# Employees with their projects

query = """
SELECT 
    e.name AS employee,
    p.project_name,
    ep.role,
    p.budget
FROM employees e
INNER JOIN employee_projects ep ON e.emp_id = ep.emp_id
INNER JOIN projects p ON ep.project_id = p.project_id
ORDER BY e.name, p.project_name
"""

print_results(query, "INNER JOIN: Employees with Projects")
print("\nNote: Only employees assigned to projects are shown!")

# 4. LEFT JOIN (LEFT OUTER JOIN)

Returns **all** rows from the left table, and matching rows from the right table. If no match, NULL values for right table columns.

```
Table A         Table B
+---+          +---+
| 1 |          | 1 |  <- Match
| 2 |          | 3 |  <- A.2 has no match
| 3 |          | 4 |  <- Match
+---+          +---+

LEFT JOIN Result: 1, 2 (NULL), 3 (all from A)
```

## Syntax
```sql
SELECT columns
FROM table1
LEFT JOIN table2 ON table1.column = table2.column
```

In [ ]:
# LEFT JOIN: ALL employees, even those without a department

query = """
SELECT 
    e.emp_id,
    e.name,
    d.dept_name,
    e.salary
FROM employees e
LEFT JOIN departments d ON e.dept_id = d.dept_id
"""

print_results(query, "LEFT JOIN: ALL Employees (with or without dept)")
print("\nNote: Henry Taylor is included with NULL department!")

In [ ]:
# LEFT JOIN: Find employees WITHOUT projects

query = """
SELECT 
    e.name,
    ep.project_id
FROM employees e
LEFT JOIN employee_projects ep ON e.emp_id = ep.emp_id
WHERE ep.project_id IS NULL
"""

print_results(query, "Employees WITHOUT any projects")

In [ ]:
# LEFT JOIN: All departments with employee count

query = """
SELECT 
    d.dept_name,
    COUNT(e.emp_id) as employee_count
FROM departments d
LEFT JOIN employees e ON d.dept_id = e.dept_id
GROUP BY d.dept_id, d.dept_name
"""

print_results(query, "Departments with Employee Count")
print("\nNote: Research shows 0 - it has no employees!")

# 5. RIGHT JOIN (RIGHT OUTER JOIN)

Returns **all** rows from the right table, and matching rows from the left table. If no match, NULL values for left table columns.

```
Table A         Table B
+---+          +---+
| 1 |          | 1 |  <- Match
| 2 |          | 3 |  <- Match
| 3 |          | 4 |  <- B.4 has no match
+---+          +---+

RIGHT JOIN Result: 1, 3, 4 (NULL) (all from B)
```

**Note:** RIGHT JOIN is less commonly used. You can usually rewrite it as a LEFT JOIN by swapping tables.

In [ ]:
# RIGHT JOIN: ALL departments, even those without employees

query = """
SELECT 
    d.dept_name,
    e.name
FROM employees e
RIGHT JOIN departments d ON e.dept_id = d.dept_id
"""

print_results(query, "RIGHT JOIN: ALL Departments")
print("\nNote: Research dept is included with NULL employee!")

In [ ]:
# Same result using LEFT JOIN (preferred style)

query = """
SELECT 
    d.dept_name,
    e.name
FROM departments d
LEFT JOIN employees e ON d.dept_id = e.dept_id
"""

print_results(query, "Same result using LEFT JOIN (swapped tables)")

# 6. FULL OUTER JOIN

Returns **all** rows when there's a match in **either** table. MySQL doesn't support FULL OUTER JOIN directly, but we can simulate it.

```
Table A         Table B
+---+          +---+
| 1 |          | 1 |  <- Match
| 2 |          | 3 |  <- Match
| 3 |          | 4 |  <- B only
+---+          +---+
  ^-- A.2 only

FULL OUTER JOIN Result: 1, 2 (NULL), 3, 4 (NULL)
```

In [ ]:
# FULL OUTER JOIN simulation using UNION
# All employees AND all departments (including unmatched)

query = """
SELECT 
    e.name,
    d.dept_name
FROM employees e
LEFT JOIN departments d ON e.dept_id = d.dept_id

UNION

SELECT 
    e.name,
    d.dept_name
FROM employees e
RIGHT JOIN departments d ON e.dept_id = d.dept_id
"""

print_results(query, "FULL OUTER JOIN (simulated with UNION)")
print("\nNote: Both Henry (no dept) AND Research (no employees) included!")

# 7. CROSS JOIN

Returns the **Cartesian product** - every row from table A combined with every row from table B.

```
Table A (3 rows) x Table B (2 rows) = 6 rows

A.1 + B.1
A.1 + B.2
A.2 + B.1
A.2 + B.2
A.3 + B.1
A.3 + B.2
```

**Use with caution!** Can produce very large result sets.

In [ ]:
# CROSS JOIN: Every employee with every project combination

query = """
SELECT 
    e.name,
    p.project_name
FROM employees e
CROSS JOIN projects p
ORDER BY e.name, p.project_name
"""

print_results(query, "CROSS JOIN: All Employee-Project Combinations")
print("\n8 employees x 4 projects = 32 combinations!")

# 8. SELF JOIN

A table joins with **itself**. Useful for hierarchical data (like org charts) or comparing rows within the same table.

## Syntax
```sql
SELECT columns
FROM table1 AS a
JOIN table1 AS b ON a.column = b.column
```

In [ ]:
# SELF JOIN: Employees with their managers

query = """
SELECT 
    e.name AS employee,
    m.name AS manager
FROM employees e
LEFT JOIN employees m ON e.manager_id = m.emp_id
"""

print_results(query, "SELF JOIN: Employees and their Managers")

In [ ]:
# SELF JOIN: Find employees who earn more than their manager

query = """
SELECT 
    e.name AS employee,
    e.salary AS emp_salary,
    m.name AS manager,
    m.salary AS mgr_salary
FROM employees e
INNER JOIN employees m ON e.manager_id = m.emp_id
WHERE e.salary > m.salary
"""

print_results(query, "Employees earning more than their manager")

In [ ]:
# SELF JOIN: Find employees in the same department

query = """
SELECT 
    e1.name AS employee1,
    e2.name AS employee2,
    d.dept_name
FROM employees e1
INNER JOIN employees e2 ON e1.dept_id = e2.dept_id AND e1.emp_id < e2.emp_id
INNER JOIN departments d ON e1.dept_id = d.dept_id
ORDER BY d.dept_name
"""

print_results(query, "Employee pairs in same department")
print("\nNote: e1.emp_id < e2.emp_id avoids duplicate pairs!")

# 9. Join Visualizations

```
INNER JOIN:                LEFT JOIN:                 RIGHT JOIN:
    A     B                    A     B                    A     B
  +---+ +---+                +---+ +---+                +---+ +---+
  |   |X|   |                |XXX|X|   |                |   |X|XXX|
  |   |X|   |                |XXX|X|   |                |   |X|XXX|
  +---+ +---+                +---+ +---+                +---+ +---+
  Only overlap              All A + overlap            All B + overlap


FULL OUTER JOIN:           CROSS JOIN:                
    A     B                    A x B = Every combination
  +---+ +---+                
  |XXX|X|XXX|                  A: [1, 2]    B: [a, b]
  |XXX|X|XXX|                  Result: [(1,a), (1,b), (2,a), (2,b)]
  +---+ +---+                
  All A + All B
```

# 10. Complex Join Examples

In [ ]:
# Complex Query: Employee details with department, manager, and project count

query = """
SELECT 
    e.name AS employee,
    d.dept_name AS department,
    m.name AS manager,
    e.salary,
    COUNT(ep.project_id) AS project_count
FROM employees e
LEFT JOIN departments d ON e.dept_id = d.dept_id
LEFT JOIN employees m ON e.manager_id = m.emp_id
LEFT JOIN employee_projects ep ON e.emp_id = ep.emp_id
GROUP BY e.emp_id, e.name, d.dept_name, m.name, e.salary
ORDER BY e.name
"""

print_results(query, "Complete Employee Report")

In [ ]:
# Find projects with total budget and employee count

query = """
SELECT 
    p.project_name,
    p.budget,
    COUNT(ep.emp_id) AS team_size,
    GROUP_CONCAT(e.name SEPARATOR ', ') AS team_members
FROM projects p
LEFT JOIN employee_projects ep ON p.project_id = ep.project_id
LEFT JOIN employees e ON ep.emp_id = e.emp_id
GROUP BY p.project_id, p.project_name, p.budget
"""

print_results(query, "Project Teams")

In [ ]:
# Department statistics with aggregations

query = """
SELECT 
    d.dept_name,
    COUNT(e.emp_id) AS employee_count,
    COALESCE(AVG(e.salary), 0) AS avg_salary,
    COALESCE(SUM(e.salary), 0) AS total_salary,
    COALESCE(MAX(e.salary), 0) AS max_salary
FROM departments d
LEFT JOIN employees e ON d.dept_id = e.dept_id
GROUP BY d.dept_id, d.dept_name
ORDER BY total_salary DESC
"""

print_results(query, "Department Statistics")

# 11. Python Helper Functions for Joins

In [ ]:
import mysql.connector

class JoinQueries:
    """Helper class for common join operations."""
    
    def __init__(self, config):
        self.config = config
    
    def _execute(self, query, params=None):
        """Execute query and return results."""
        with mysql.connector.connect(**self.config) as conn:
            with conn.cursor(dictionary=True) as cursor:
                cursor.execute(query, params or ())
                return cursor.fetchall()
    
    def employees_with_departments(self):
        """Get all employees with their department names."""
        return self._execute("""
            SELECT e.*, d.dept_name
            FROM employees e
            LEFT JOIN departments d ON e.dept_id = d.dept_id
        """)
    
    def employees_with_managers(self):
        """Get employees with their manager names."""
        return self._execute("""
            SELECT e.name AS employee, m.name AS manager
            FROM employees e
            LEFT JOIN employees m ON e.manager_id = m.emp_id
        """)
    
    def employees_on_project(self, project_name):
        """Get all employees on a specific project."""
        return self._execute("""
            SELECT e.name, ep.role
            FROM employees e
            INNER JOIN employee_projects ep ON e.emp_id = ep.emp_id
            INNER JOIN projects p ON ep.project_id = p.project_id
            WHERE p.project_name = %s
        """, (project_name,))
    
    def department_summary(self):
        """Get department statistics."""
        return self._execute("""
            SELECT 
                d.dept_name,
                COUNT(e.emp_id) AS emp_count,
                COALESCE(AVG(e.salary), 0) AS avg_salary
            FROM departments d
            LEFT JOIN employees e ON d.dept_id = e.dept_id
            GROUP BY d.dept_id, d.dept_name
        """)


# Example usage:
# jq = JoinQueries(DB_CONFIG)
# print(jq.employees_with_departments())
# print(jq.employees_on_project("Mobile App"))

print("JoinQueries class defined!")

# 12. Summary

## Join Types Quick Reference

| Join | Returns | Use When |
|------|---------|----------|
| `INNER JOIN` | Only matching rows | Need data that exists in BOTH tables |
| `LEFT JOIN` | All left + matching right | Need ALL from main table + related data |
| `RIGHT JOIN` | All right + matching left | (Use LEFT JOIN instead - swap tables) |
| `FULL OUTER JOIN` | All from both tables | Need complete data from both sides |
| `CROSS JOIN` | Cartesian product | Generate all combinations |
| `SELF JOIN` | Table with itself | Hierarchies, comparisons within table |

## Common Patterns

| Pattern | Query |
|---------|-------|
| Find unmatched | `LEFT JOIN ... WHERE right.id IS NULL` |
| Employee-Manager | `SELF JOIN ON e.manager_id = m.emp_id` |
| Many-to-Many | Join through junction table |
| Count by category | `LEFT JOIN + GROUP BY + COUNT()` |

## Best Practices

1. **Use table aliases** (`e`, `d`, `m`) for readability
2. **Prefer LEFT JOIN over RIGHT JOIN** - more intuitive
3. **Index foreign key columns** - improves join performance
4. **Be careful with CROSS JOIN** - can explode result size
5. **Use INNER JOIN when NULL values aren't needed** - better performance